In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
import math
import numpy as np

import models.merton_jumps as mj
from utils.plotting import plot_function_vs_param

In [3]:
test_params = {
    "V": 130,
    "D": 100,
    "r": 0.02,
    "T": 1,
    "sigma_V": 0.3,
    "lambda_": 0.5,
    "gamma": math.log(1-0.25),
    "delta": 0.2
}
mj.equity_value(**test_params)

np.float64(38.09890818986286)

In [ ]:
from functools import partial
import sims.engine as mce
import sims.gjd as gjd 

jump_dist = gjd.lognormal_jump_dist(delta=test_params["delta"], gamma=test_params["gamma"])
# jump_dist = gjd.double_exponential_jump_dist(p=0.5, eta1=2, eta2=0.5)
# jump_dist = gjd.constant_jump_dist(-0.5)

step_fn = partial(
    gjd.step,
    mu_total=test_params["r"],
    sigma=test_params["sigma_V"],
    lambda_=test_params["lambda_"],
    jump_dist=jump_dist,
)

times = np.linspace(0, test_params["T"], 2)
n_paths = 100_000

events = {
    test_params["T"]: {
        "equity_payoff": lambda V: np.maximum(V - test_params["D"], 0),
    }
}

results = mce.run_monte_carlo(
    initial_state_fn=partial(gjd.initial_state, S0=test_params["V"]),
    step_fn=step_fn,
    times=times,
    n_paths=n_paths,
    events=events,
    seed=42,
)
equity_value = math.exp(-test_params["r"] * test_params["T"]) * np.mean(results["equity_payoff"])
equity_value

np.float64(43.73691011818534)

In [5]:
fig = plot_function_vs_param(
    mj.equity_value,
    x_param="V",
    x_values=np.linspace(50, 200),
    base_params={
        "D": 100,
        "r": 0.02,
        "T": 1,
        "sigma_V": 0.3,
        "gamma": math.log(1 - 0.25),
        "delta": 0.2 
    },
    y_title="E(V)",
    varying_param="lambda_",
    varying_values=[0.0, 0.1, 0.5],
    title="Equity value"
)

fig.show()

In [6]:
fig = plot_function_vs_param(
    mj.credit_spread,
    x_param="T",
    x_values=np.linspace(0, 5, 1000),
    base_params={
        "V": 130,
        "D": 100,
        "r": 0.02,
        "sigma_V": 0.3,
        "gamma": math.log(1 - 0.25),
        "delta": 0.2
    },
    y_title="s(T)",
    varying_param="lambda_",
    varying_values=[0.0, 0.5, 1.0],
    title="Credit spread"
)

fig.show()